# S02 — Fundamentos PySpark
## Proyecto Sello: Observatorio domiciliario del servicio de agua de Juliaca

**Nombre:**
**Equipo:** Horeb
**Sesión:** S02 - Fundamentos PySpark: transformaciones, funciones, agrupaciones y evaluación perezosa
**Rol o aporte realizado:**
**Link de GitHub:**

---

> **Nota sobre los datos.** La consigna permite trabajar sobre una muestra
> representativa del caso de negocio definido en S1 cuando el equipo todavía no
> dispone de datos reales. Los nodos físicos están en fase de adquisición, así
> que este notebook opera sobre una muestra generada con la física y los
> parámetros del caso: 8 tanques en 3 zonas de Juliaca, 30 días de registro cada
> 10 minutos, con suministro intermitente por zona, ciclo térmico del altiplano
> y pérdidas por fuga en tres viviendas.

---
## 1. Extracción y exploración del DataFrame

*Equivalente a 3.3 y 3.4 de la guía.*

In [20]:
from pyspark.sql import SparkSession
from pyspark.sql.functions import (
    col, when, lit, hour, to_timestamp,
    avg, min as smin, max as smax, count, sum as ssum, round as sround
)

spark = (SparkSession.builder
         .appName("S02_Horeb_MonitorAgua")
         .master("local[*]")
         .getOrCreate())
spark.sparkContext.setLogLevel("ERROR")

print("Spark", spark.version)

Spark 4.2.0


In [21]:
# El CSV debe estar en la misma carpeta que este notebook
RUTA = "../tanques_horeb_muestra.csv"

df = spark.read.csv(RUTA, header=True, inferSchema=True)
df.printSchema()

root
 |-- device_id: string (nullable = true)
 |-- timestamp: timestamp (nullable = true)
 |-- zona: string (nullable = true)
 |-- distancia_cm: double (nullable = true)
 |-- temp_aire_c: double (nullable = true)
 |-- hum_aire_pct: double (nullable = true)
 |-- nivel_cm_comp: double (nullable = true)
 |-- litros: double (nullable = true)
 |-- capacidad_litros: integer (nullable = true)
 |-- altura_tanque_cm: integer (nullable = true)
 |-- presion_linea_kpa: double (nullable = true)
 |-- caudal_lpm: double (nullable = true)
 |-- temp_agua_c: double (nullable = true)
 |-- orp_mv: double (nullable = true)
 |-- horas_residencia: double (nullable = true)
 |-- bateria_pct: double (nullable = true)



In [22]:
print(f"Filas:    {df.count():,}")
print(f"Columnas: {len(df.columns)}")
print(f"Tanques:  {df.select('device_id').distinct().count()}")
print(f"Zonas:    {df.select('zona').distinct().count()}")
df.show(5, truncate=False)

Filas:    34,560
Columnas: 16
Tanques:  8
Zonas:    3
+---------------+-------------------+------+------------+-----------+------------+-------------+------+----------------+----------------+-----------------+----------+-----------+------+----------------+-----------+
|device_id      |timestamp          |zona  |distancia_cm|temp_aire_c|hum_aire_pct|nivel_cm_comp|litros|capacidad_litros|altura_tanque_cm|presion_linea_kpa|caudal_lpm|temp_agua_c|orp_mv|horas_residencia|bateria_pct|
+---------------+-------------------+------+------------+-----------+------------+-------------+------+----------------+----------------+-----------------+----------+-----------+------+----------------+-----------+
|horeb_master_01|2026-07-15 00:00:00|Zona A|46.7        |-1.08      |98.2        |55.01        |550.0 |1000            |100             |0.0              |0.0       |8.91       |619.2 |0.17            |100.0      |
|horeb_master_01|2026-07-15 00:10:00|Zona A|47.17       |0.4        |96.2        |54.4

In [23]:
# Verificación explícita de inconsistencias (nivel A de la rúbrica)
nulos   = df.filter(col("litros").isNull()).count()
fuera   = df.filter(col("distancia_cm") > col("altura_tanque_cm")).count()
negativo= df.filter(col("nivel_cm_comp") < 0).count()

print(f"Lecturas sin nivel (caída de red):      {nulos}")
print(f"Distancia mayor que la altura del tanque: {fuera}")
print(f"Nivel compensado negativo:               {negativo}")

df.select("distancia_cm", "temp_aire_c", "nivel_cm_comp", "litros").describe().show()

Lecturas sin nivel (caída de red):      207
Distancia mayor que la altura del tanque: 546
Nivel compensado negativo:               13
+-------+------------------+------------------+------------------+------------------+
|summary|      distancia_cm|       temp_aire_c|     nivel_cm_comp|            litros|
+-------+------------------+------------------+------------------+------------------+
|  count|             34560|             34560|             34353|             34353|
|   mean| 32.08764154038251|12.999168981481482| 73.77125840537987|  791.569440805752|
| stddev|26.472510786722374|10.621312239931317|22.985761338490835|264.59928116604647|
|    min|               0.5|             -4.02|             -0.31|               0.0|
|    max|483.22390473665246|             30.06|            119.51|            1494.0|
+-------+------------------+------------------+------------------+------------------+



**Explicación:**

*(Describe aquí qué muestra el esquema, cuántas filas cargaste y qué
inconsistencias encontraste. Las lecturas nulas corresponden a cortes de
transmisión; las distancias fuera de rango son ecos espurios por oleaje o espuma
durante el llenado.)*

---
## 2. Transformaciones, funciones y evaluación perezosa

*Equivalente a 3.5 y 3.7 de la guía.*

Primero se encadenan las transformaciones. Nada se ejecuta todavía.

In [24]:
import time

t0 = time.time()

# --- Transformación 1: conversión de tipo con to_timestamp
d1 = df.withColumn("ts", to_timestamp(col("timestamp")))

# --- Transformación 2: filtro de lecturas válidas
d2 = d1.filter(col("litros").isNotNull()) \
       .filter(col("distancia_cm") <= col("altura_tanque_cm"))

# --- Transformación 3: hora del día
d3 = d2.withColumn("hora", hour(col("ts")))

t1 = time.time()
print(f"Tiempo de las 3 transformaciones: {t1 - t0:.4f} s")
print("Spark no leyó ni una fila todavía: solo construyó el plan lógico.")

Tiempo de las 3 transformaciones: 0.0563 s
Spark no leyó ni una fila todavía: solo construyó el plan lógico.


In [25]:
# Ahora sí, una ACCIÓN. Aquí es donde Spark ejecuta todo el plan anterior.
t0 = time.time()
n = d3.count()
t1 = time.time()

print(f"Filas válidas: {n:,}")
print(f"Tiempo de la acción count(): {t1 - t0:.4f} s")
print("\nLa diferencia entre ambos tiempos es la evaluación perezosa.")

Filas válidas: 33,810
Tiempo de la acción count(): 0.2112 s

La diferencia entre ambos tiempos es la evaluación perezosa.


In [26]:
# --- Funciones de columna: withColumn, lit, cast y when

# Nivel tal como lo reportaría el sensor SIN compensar la velocidad del sonido
d4 = d3.withColumn(
    "nivel_crudo_cm",
    col("altura_tanque_cm").cast("double") - col("distancia_cm").cast("double")
)

# Error introducido por la deriva térmica
d5 = d4.withColumn("error_termico_cm",
                   sround(col("nivel_cm_comp") - col("nivel_crudo_cm"), 3))

# Porcentaje de llenado y clasificación con when / otherwise / lit
d6 = d5.withColumn("pct_llenado", sround(col("litros") / col("capacidad_litros"), 3))

datos = d6.withColumn(
    "estado",
    when(col("pct_llenado") < lit(0.15), lit("critico"))
    .when(col("pct_llenado") < lit(0.40), lit("bajo"))
    .otherwise(lit("normal"))
)

datos.select("device_id", "hora", "nivel_crudo_cm", "nivel_cm_comp",
             "error_termico_cm", "pct_llenado", "estado").show(8)

+---------------+----+--------------+-------------+----------------+-----------+------+
|      device_id|hora|nivel_crudo_cm|nivel_cm_comp|error_termico_cm|pct_llenado|estado|
+---------------+----+--------------+-------------+----------------+-----------+------+
|horeb_master_01|   0|          53.3|        55.01|            1.71|       0.55|normal|
|horeb_master_01|   0|         52.83|        54.43|             1.6|      0.544|normal|
|horeb_master_01|   0|         53.13|        54.87|            1.74|      0.549|normal|
|horeb_master_01|   0|          53.0|         54.7|             1.7|      0.547|normal|
|horeb_master_01|   0|         52.73|        54.49|            1.76|      0.545|normal|
|horeb_master_01|   0|         53.31|        55.02|            1.71|       0.55|normal|
|horeb_master_01|   1|         53.42|        55.18|            1.76|      0.552|normal|
|horeb_master_01|   1|         52.64|        54.45|            1.81|      0.544|normal|
+---------------+----+----------

In [27]:
datos.cache()
datos.groupBy("estado").count().orderBy("count", ascending=False).show()

[Stage 34:>                                                         (0 + 1) / 1]

+-------+-----+
| estado|count|
+-------+-----+
| normal|30788|
|   bajo| 2456|
|critico|  566|
+-------+-----+



**Explicación:**

*(Explica aquí las tres transformaciones aplicadas, las funciones de columna
usadas — `withColumn`, `when`, `lit`, `.cast()` — y cómo los dos tiempos medidos
demuestran la evaluación perezosa: las transformaciones tardan milésimas porque
solo construyen el DAG; la acción `count()` es la que dispara la ejecución
real.)*

---
## 3. Agrupaciones y agregaciones

*Equivalente a 3.9 de la guía.*

In [28]:
# Resumen por zona: cuánta agua tiene cada sector y qué tan frío es
datos.groupBy("zona").agg(
    count("*").alias("lecturas"),
    sround(avg("litros"), 1).alias("litros_prom"),
    sround(avg("pct_llenado") * 100, 1).alias("llenado_prom_pct"),
    sround(smin("temp_aire_c"), 1).alias("temp_aire_min"),
    sround(smax("temp_aire_c"), 1).alias("temp_aire_max"),
).orderBy("zona").show()

+------+--------+-----------+----------------+-------------+-------------+
|  zona|lecturas|litros_prom|llenado_prom_pct|temp_aire_min|temp_aire_max|
+------+--------+-----------+----------------+-------------+-------------+
|Zona A|   12736|      820.0|            70.4|         -3.6|         29.7|
|Zona B|    8389|      803.3|            76.9|         -4.0|         30.1|
|Zona C|   12685|      786.3|            69.8|         -3.7|         29.9|
+------+--------+-----------+----------------+-------------+-------------+



In [29]:
# Caudal mínimo nocturno: el indicador estándar de detección de fugas.
# Entre las 2 y las 4 de la madrugada no debería haber consumo.
madrugada = datos.filter(col("hora").between(2, 4))

madrugada.groupBy("device_id", "zona").agg(
    sround(avg("litros"), 1).alias("litros_prom_madrugada"),
    sround(avg("error_termico_cm"), 3).alias("error_termico_prom"),
    count("*").alias("lecturas"),
).orderBy("device_id").show()

+---------------+------+---------------------+------------------+--------+
|      device_id|  zona|litros_prom_madrugada|error_termico_prom|lecturas|
+---------------+------+---------------------+------------------+--------+
|horeb_master_01|Zona A|                554.4|             1.657|     477|
|      horeb_n02|Zona A|                586.3|             1.533|     537|
|      horeb_n03|Zona A|                803.3|             2.065|     538|
|      horeb_n04|Zona B|                738.1|             1.067|     537|
|      horeb_n05|Zona B|                618.3|             1.621|     481|
|      horeb_n06|Zona C|                621.6|             1.405|     534|
|      horeb_n07|Zona C|                730.8|             2.283|     496|
|      horeb_n08|Zona C|                801.9|             0.866|     535|
+---------------+------+---------------------+------------------+--------+



In [30]:
# El error térmico depende de la hora, no del tanque: es una firma diurna
datos.groupBy("hora").agg(
    sround(avg("temp_aire_c"), 1).alias("temp_aire_prom"),
    sround(avg("error_termico_cm"), 3).alias("error_termico_cm"),
).orderBy("hora").show(24)

+----+--------------+----------------+
|hora|temp_aire_prom|error_termico_cm|
+----+--------------+----------------+
|   0|          -0.7|           1.549|
|   1|          -1.8|             1.6|
|   2|          -1.9|           1.652|
|   3|          -1.0|           1.578|
|   4|           0.9|            1.43|
|   5|           3.6|           1.117|
|   6|           7.0|           0.696|
|   7|          10.7|           0.458|
|   8|          14.6|           0.322|
|   9|          18.4|           0.118|
|  10|          21.9|           0.064|
|  11|          24.7|          -0.048|
|  12|          26.7|          -0.134|
|  13|          27.8|          -0.173|
|  14|          27.8|          -0.179|
|  15|          26.9|           -0.26|
|  16|          25.1|          -0.105|
|  17|          22.4|          -0.046|
|  18|          19.0|           0.187|
|  19|          15.3|           0.384|
|  20|          11.4|           0.607|
|  21|           7.6|           1.037|
|  22|           4.1|    

**Explicación:**

*(Interpreta los resultados: qué zona tiene menor llenado promedio, y sobre todo
la última tabla — el error térmico es máximo de madrugada y casi nulo al
mediodía, porque el sensor está calibrado para 20 °C.)*

---
## 4. RDD y plan de ejecución

*Equivalente a 3.6 y 3.10 de la guía.*

In [31]:
# Plan de ejecución: filtro seguido de proyección
consulta = df.filter(col("zona") == "Zona A").select("device_id", "litros")
consulta.explain(True)

== Parsed Logical Plan ==
'Project ['device_id, 'litros]
+- Filter (zona#4204 = Zona A)
   +- Relation [device_id#4202,timestamp#4203,zona#4204,distancia_cm#4205,temp_aire_c#4206,hum_aire_pct#4207,nivel_cm_comp#4208,litros#4209,capacidad_litros#4210,altura_tanque_cm#4211,presion_linea_kpa#4212,caudal_lpm#4213,temp_agua_c#4214,orp_mv#4215,horas_residencia#4216,bateria_pct#4217] csv

== Analyzed Logical Plan ==
device_id: string, litros: double
Project [device_id#4202, litros#4209]
+- Filter (zona#4204 = Zona A)
   +- Relation [device_id#4202,timestamp#4203,zona#4204,distancia_cm#4205,temp_aire_c#4206,hum_aire_pct#4207,nivel_cm_comp#4208,litros#4209,capacidad_litros#4210,altura_tanque_cm#4211,presion_linea_kpa#4212,caudal_lpm#4213,temp_agua_c#4214,orp_mv#4215,horas_residencia#4216,bateria_pct#4217] csv

== Optimized Logical Plan ==
Project [device_id#4202, litros#4209]
+- Filter (isnotnull(zona#4204) AND (zona#4204 = Zona A))
   +- Relation [device_id#4202,timestamp#4203,zona#4204,distan

**Optimización de Catalyst identificada: *predicate pushdown*.**

En el plan físico aparece `PushedFilters: [IsNotNull(zona), EqualTo(zona,Zona A)]`.
Catalyst movió el filtro por zona hasta la propia lectura del archivo, de modo
que las filas de las otras zonas nunca llegan a materializarse en memoria.
Además, en `ReadSchema` se ve que solo lee tres columnas de las dieciséis
(*column pruning*), porque son las únicas que la consulta necesita.

*(Comparar con el Parsed Logical Plan, donde el filtro aparecía por encima de la
proyección: ese reordenamiento es lo que hace Catalyst.)*

In [32]:
# --- Operaciones sobre RDD
rdd = datos.select("zona", "device_id", "litros", "estado", "hora").rdd
print("Tipo:", type(rdd))
print("Primer elemento:", rdd.first())

Tipo: <class 'pyspark.core.rdd.RDD'>


[Stage 47:>                                                         (0 + 1) / 1]

Primer elemento: Row(zona='Zona A', device_id='horeb_master_01', litros=550.0, estado='normal', hora=0)


In [33]:
# map + reduceByKey: litros acumulados por zona
pares    = rdd.map(lambda r: (r["zona"], float(r["litros"])))
por_zona = pares.reduceByKey(lambda a, b: a + b)
print("Litros acumulados por zona:")
for z, v in sorted(por_zona.collect()):
    print(f"  {z}: {v:,.0f}")

Litros acumulados por zona:
  Zona A: 10,443,675
  Zona B: 6,739,238
  Zona C: 9,973,635


In [34]:
# filter + map + reduceByKey: lecturas en estado crítico por tanque
criticos = (rdd
            .filter(lambda r: r["estado"] == "critico")
            .map(lambda r: (r["device_id"], 1))
            .reduceByKey(lambda a, b: a + b))

print("Lecturas en estado crítico (tanque bajo 15%):")
for k, v in sorted(criticos.collect(), key=lambda x: -x[1]):
    print(f"  {k}: {v}")

Lecturas en estado crítico (tanque bajo 15%):
  horeb_n07: 266
  horeb_n06: 97
  horeb_master_01: 70
  horeb_n03: 70
  horeb_n05: 63


In [35]:
# flatMap: descomponer cada lectura en pares (dimensión, valor)
etiquetas = (rdd
             .flatMap(lambda r: [("zona:" + r["zona"], 1), ("estado:" + r["estado"], 1)])
             .reduceByKey(lambda a, b: a + b))
for k, v in sorted(etiquetas.collect()):
    print(f"  {k}: {v:,}")

  estado:bajo: 2,456
  estado:critico: 566
  estado:normal: 30,788
  zona:Zona A: 12,736
  zona:Zona B: 8,389
  zona:Zona C: 12,685


**Explicación:**

*(Explica qué hace cada operación RDD y por qué en la práctica conviene usar
DataFrame: el RDD no pasa por Catalyst, así que estas mismas agregaciones no
reciben predicate pushdown ni column pruning.)*

---
## 5. Error o hallazgo

**Qué ocurrió.** Al revisar la señal de nivel encontré que el sensor ultrasónico
reporta una caída de nivel durante la madrugada aun cuando no hay consumo. El
HC-SR04 calcula la distancia asumiendo una velocidad del sonido fija, la de
20 °C. Pero la velocidad del sonido en aire depende de la temperatura según

$$c = 331.3 \cdot \sqrt{1 + T/273.15}$$

Dentro de un tanque en una azotea de Juliaca, la columna de aire recorre de
−2 °C de madrugada a cerca de 28 °C al mediodía. Eso hace variar la velocidad
del sonido entre 330.1 y 349.0 m/s, un 5.7 %. Como el sensor asume 343.2 m/s,
de madrugada **sobreestima la distancia y por lo tanto subestima el nivel**.

**Por qué importa.** El artefacto baja de forma monótona toda la noche y toca su
mínimo al amanecer: tiene exactamente la forma de una fuga lenta, justo en la
ventana horaria donde el sistema busca fugas. Una fuga real de flotador pierde
entre 20 y 60 litros por noche, del mismo orden que el error térmico.

**Cómo lo identifiqué.** Agrupando el error por hora del día (bloque 3): el error
es máximo entre las 2 y las 4 de la madrugada y prácticamente nulo cerca del
mediodía, y sigue la temperatura, no el consumo.

**Qué decidí.** Registrar en el mismo evento tanto la lectura cruda
(`distancia_cm`) como el nivel compensado (`nivel_cm_comp`), y compensar la
velocidad del sonido con la temperatura y la humedad medidas dentro de la propia
cámara de aire. La detección de fugas opera únicamente sobre la señal
compensada. Conservar el dato crudo permite volver atrás si aparece otro efecto
no explicado.

In [36]:
# Evidencia del hallazgo
import pandas as pd

ev = (datos.groupBy("hora")
      .agg(sround(avg("temp_aire_c"), 2).alias("temp_aire_c"),
           sround(avg("error_termico_cm"), 3).alias("error_cm"))
      .orderBy("hora").toPandas())

ev["error_litros_aprox"] = (ev["error_cm"] * 10).round(1)   # tanque de 1000 L / 100 cm
print(ev.to_string(index=False))

/usr/local/lib/python3.10/site-packages/pyspark/sql/pandas/conversion.py:348: UserWarning: toPandas attempted Arrow optimization because 'spark.sql.execution.arrow.pyspark.enabled' is set to true; however, failed by the reason below:
  [PACKAGE_NOT_INSTALLED] PyArrow >= 18.0.0 must be installed; however, it was not found.
Attempting non-optimization as 'spark.sql.execution.arrow.pyspark.fallback.enabled' is set to true.
  warn(msg)


 hora  temp_aire_c  error_cm  error_litros_aprox
    0        -0.66     1.549                15.5
    1        -1.80     1.600                16.0
    2        -1.87     1.652                16.5
    3        -0.95     1.578                15.8
    4         0.94     1.430                14.3
    5         3.64     1.117                11.2
    6         6.98     0.696                 7.0
    7        10.73     0.458                 4.6
    8        14.62     0.322                 3.2
    9        18.44     0.118                 1.2
   10        21.86     0.064                 0.6
   11        24.67    -0.048                -0.5
   12        26.67    -0.134                -1.3
   13        27.79    -0.173                -1.7
   14        27.85    -0.179                -1.8
   15        26.94    -0.260                -2.6
   16        25.06    -0.105                -1.0
   17        22.36    -0.046                -0.5
   18        19.02     0.187                 1.9
   19        15.29  

---
## 6. Reflexión técnica

**¿Por qué la evaluación perezosa es útil para procesar datos a escala, y qué
riesgo tendría si Spark ejecutara cada transformación de inmediato?**

*(Responde en 5 a 8 líneas. Puntos que conviene tocar:)*

- *Spark acumula las transformaciones en un DAG y solo ejecuta cuando llega una
  acción, lo que le permite ver la consulta completa antes de decidir cómo
  correrla.*
- *Con esa vista completa, Catalyst reordena operaciones: empuja los filtros
  hasta la lectura del archivo y descarta las columnas que nadie usa.*
- *En este notebook eso significó leer 3 columnas de 16 y descartar dos zonas
  antes de materializar nada.*
- *Si cada transformación se ejecutara al escribirse, cada paso intermedio
  produciría un dataset completo en memoria o en disco.*
- *Con 26 millones de registros proyectados a seis meses, esos intermedios
  serían el cuello de botella, y ninguna optimización global sería posible
  porque Spark no sabría qué viene después.*

In [18]:
spark.stop()
print("Sesión Spark cerrada.")

Sesión Spark cerrada.
